# RelCheck — Master Evaluation Notebook
**CS298 Spring 2026 | Siddhi Patil | SJSU**

This notebook runs the **complete RelCheck pipeline** end-to-end:
- ✅ Pilot demo on 12 images
- ✅ R-Bench 200-image evaluation
- ✅ Two baselines + full RelCheck
- ✅ Ablation study (3 variants)
- ✅ R-CHAIR annotation helper
- ✅ All figures and metric tables

**How to use:** Set your API key and GitHub URL in Section 0, then click **Runtime → Run all**.

---

## Section 0: Setup
Run these cells once at the start of every Colab session.

In [ ]:
# ── GPU Check ──────────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime → Change runtime type → GPU (T4 or A100)."
    )
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")
if vram_gb < 14:
    print("⚠️  Less than 14 GB VRAM detected. Consider switching to A100 in Colab Pro.")

In [ ]:
# ── Install dependencies (run once; ~3 min) ────────────────────────────────
!pip install -q \
    spacy transformers[torch] accelerate bitsandbytes \
    together python-Levenshtein nltk \
    pandas matplotlib seaborn requests tqdm Pillow
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ All dependencies installed")

In [ ]:
# ── Clone your GitHub repo ─────────────────────────────────────────────────
GITHUB_REPO_URL = "https://github.com/siddhipatil503/RelCheck.git"
REPO_DIR = "/content/RelCheck"

import os, sys
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Add relcheck/ package to Python path
sys.path.insert(0, f"{REPO_DIR}/relcheck")

# Create output directories
os.makedirs(f"{REPO_DIR}/eval",    exist_ok=True)
os.makedirs(f"{REPO_DIR}/figures", exist_ok=True)

print(f"✅ Repo ready at {REPO_DIR}")

In [ ]:
# ── API Key ────────────────────────────────────────────────────────────────
# Paste your Together.ai key here (get one free at https://api.together.xyz)
TOGETHER_API_KEY = ""  # ← PASTE YOUR KEY HERE

import os
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
assert TOGETHER_API_KEY, "Together.ai API key is required. Paste it above."
print("✅ API key set")

---
## Section 1: Load Models
BLIP-2 is loaded once here and shared across all pipeline stages to avoid OOM errors.
Loading takes ~3–5 minutes on first run (downloading ~15 GB of weights).

In [ ]:
# ── Load BLIP-2 (8-bit quantized to fit in T4 16 GB) ──────────────────────
# BitsAndBytesConfig is required in newer transformers (replaces the old load_in_8bit=True kwarg)
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
import torch

BLIP2_MODEL_ID = "Salesforce/blip2-flan-t5-xl"
print(f"Loading {BLIP2_MODEL_ID} in 8-bit mode (saves ~7 GB VRAM)...")

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

blip2_processor = Blip2Processor.from_pretrained(BLIP2_MODEL_ID)
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
blip2_model.eval()
print(f"✅ BLIP-2 loaded | VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ── Patch VQAVerifier to reuse our already-loaded BLIP-2 ──────────────────
# This prevents the verifier from trying to load a second BLIP-2 instance.
import relation_verifier as _rv

def _patched_vqa_init(self, model_name=BLIP2_MODEL_ID):
    self.processor = blip2_processor
    self.model     = blip2_model
    self.device    = next(blip2_model.parameters()).device
    print("[VQAVerifier] Using pre-loaded BLIP-2 (shared).")

_rv.VQAVerifier.__init__ = _patched_vqa_init
print("✅ VQAVerifier patched to share BLIP-2 instance")

In [ ]:
# ── Initialize RelCheck Pipeline ──────────────────────────────────────────
from relcheck_pipeline import RelCheckPipeline

pipeline = RelCheckPipeline(together_api_key=TOGETHER_API_KEY)
torch.cuda.empty_cache()
print(f"✅ Pipeline ready | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ── Helper functions used throughout ──────────────────────────────────────
from PIL import Image

def generate_caption(image: Image.Image, max_tokens: int = 80) -> str:
    """Generate a BLIP-2 caption for an image."""
    prompt  = "Describe this image in detail."
    inputs  = blip2_processor(images=image, text=prompt, return_tensors="pt").to(
        blip2_model.device, torch.float16)
    with torch.no_grad():
        ids = blip2_model.generate(**inputs, max_new_tokens=max_tokens)
    return blip2_processor.decode(ids[0], skip_special_tokens=True).strip()

def vqa_probe(image: Image.Image, question: str) -> str:
    """Run a BLIP-2 VQA probe; returns 'yes' or 'no'."""
    inputs = blip2_processor(images=image, text=question, return_tensors="pt").to(
        blip2_model.device, torch.float16)
    with torch.no_grad():
        ids = blip2_model.generate(**inputs, max_new_tokens=10)
    answer = blip2_processor.decode(ids[0], skip_special_tokens=True).strip().lower()
    return "yes" if answer.startswith("yes") else "no"

def compute_rpope_metrics(predictions: list[dict]) -> dict:
    """
    Compute R-POPE metrics from a list of {pred, gt} dicts.
    Both pred and gt should be 'yes' or 'no' strings.
    """
    tp = sum(1 for p in predictions if p['pred'] == 'yes' and p['gt'] == 'yes')
    tn = sum(1 for p in predictions if p['pred'] == 'no'  and p['gt'] == 'no')
    fp = sum(1 for p in predictions if p['pred'] == 'yes' and p['gt'] == 'no')
    fn = sum(1 for p in predictions if p['pred'] == 'no'  and p['gt'] == 'yes')
    total = len(predictions)
    accuracy  = (tp + tn) / total if total else 0
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0
    yes_ratio = (tp + fp) / total if total else 0
    return dict(accuracy=accuracy, precision=precision, recall=recall,
                f1=f1, yes_ratio=yes_ratio, tp=tp, tn=tn, fp=fp, fn=fn, total=total)

print("✅ Helper functions ready")

---
## Section 2: Pilot Demo (12 Images)
Runs the full pipeline on the 12 images in `images/`. 
This is a qualitative sanity check and produces `eval/pilot_results.csv`.

In [ ]:
import glob
IMAGES_DIR  = f"{REPO_DIR}/images"
image_paths = sorted(
    glob.glob(f"{IMAGES_DIR}/*.jpeg") +
    glob.glob(f"{IMAGES_DIR}/*.jpg")  +
    glob.glob(f"{IMAGES_DIR}/*.webp") +
    glob.glob(f"{IMAGES_DIR}/*.png")
)
print(f"Found {len(image_paths)} pilot images")

PILOT_CSV = f"{REPO_DIR}/eval/pilot_results.csv"
pilot_results = pipeline.run_batch(
    image_paths      = image_paths,
    blip2_processor  = blip2_processor,
    blip2_model      = blip2_model,
    output_csv       = PILOT_CSV,
)
print(f"\n✅ Pilot complete! {len(pilot_results)} images processed.")
print(f"Results saved → {PILOT_CSV}")

In [ ]:
import pandas as pd
df_pilot = pd.read_csv(PILOT_CSV)

print("=" * 80)
print("PILOT RESULTS SUMMARY")
print("=" * 80)
for _, row in df_pilot.iterrows():
    fname = row['image_path'].split('/')[-1]
    changed = row['original_caption'] != row['corrected_caption']
    print(f"\n[{fname}]")
    print(f"  ORIGINAL:  {row['original_caption']}")
    if changed:
        print(f"  CORRECTED: {row['corrected_caption']}")
    else:
        print(f"  (no correction needed)")
    print(f"  triples={row['n_triples']}, hallucinated={row['n_hallucinated']}, edit_rate={row['edit_rate']:.3f}")

print("\n" + "=" * 80)
print(f"Total hallucinations detected: {df_pilot['n_hallucinated'].sum()} "
      f"across {df_pilot['n_triples'].sum()} triples")
print(f"Images with ≥1 hallucination: {df_pilot['any_hallucination'].sum()}/{len(df_pilot)}")
print(f"Avg edit rate: {df_pilot['edit_rate'].mean():.3f}")

---
## Section 3: R-Bench Setup (200-Image Subset)
Downloads R-Bench annotations and a 200-image subset of nocaps validation images.
**This only needs to run once** — results are cached in `/content/rbench/`.

In [ ]:
RBENCH_DIR = "/content/R-Bench"
if not os.path.exists(RBENCH_DIR):
    !git clone https://github.com/mrwu-mac/R-Bench {RBENCH_DIR}
    print("✅ R-Bench cloned")
else:
    print("✅ R-Bench already present")

# Explore the repo to find annotation files
import pathlib
json_files = list(pathlib.Path(RBENCH_DIR).rglob("*.json"))
print(f"\nFound {len(json_files)} JSON files in R-Bench repo:")
for f in json_files[:20]:
    print(f"  {f.relative_to(RBENCH_DIR)}")

In [ ]:
import json, pathlib

# ── Auto-detect R-Bench annotation file ───────────────────────────────────
# Try common locations; update ANNOTATION_PATH manually if needed
candidates = [
    f"{RBENCH_DIR}/data/r_bench.json",
    f"{RBENCH_DIR}/data/annotations.json",
    f"{RBENCH_DIR}/annotations/r_bench.json",
    f"{RBENCH_DIR}/r_bench.json",
]
candidates += [str(f) for f in pathlib.Path(RBENCH_DIR).rglob("*.json")]

rbench_data = None
ANNOTATION_PATH = None

for cand in candidates:
    if os.path.exists(cand):
        try:
            with open(cand) as f:
                data = json.load(f)
            # Accept if it looks like a list of QA dicts
            if isinstance(data, list) and len(data) > 100 and 'question' in str(data[0]):
                rbench_data = data
                ANNOTATION_PATH = cand
                print(f"✅ Loaded R-Bench annotations from: {cand}")
                print(f"   Total questions: {len(data):,}")
                print(f"   Sample entry keys: {list(data[0].keys())}")
                break
        except Exception as e:
            pass

if rbench_data is None:
    # Manual fallback: print all JSON files so user can identify the right one
    print("⚠️  Could not auto-detect annotation file. Available JSONs:")
    for f in pathlib.Path(RBENCH_DIR).rglob("*.json"):
        size_kb = f.stat().st_size // 1024
        print(f"  {f} ({size_kb} KB)")
    print("\nManually set ANNOTATION_PATH below and re-run this cell.")
    ANNOTATION_PATH = ""  # ← set manually if needed
    with open(ANNOTATION_PATH) as f:
        rbench_data = json.load(f)

In [ ]:
# ── Normalize R-Bench entries to a consistent schema ──────────────────────
# Handles different possible key names in R-Bench JSON
def normalize_entry(entry: dict) -> dict:
    """Normalize a R-Bench entry to {image_id, image_url, question, answer, relation_type}."""
    def get(keys, default=None):
        for k in keys:
            if k in entry: return entry[k]
        return default

    return {
        "image_id":      get(["image_id", "img_id", "id"], str(hash(str(entry)))),
        "image_url":     get(["image_url", "url", "img_url"], ""),
        "question":      get(["question", "q", "text"], ""),
        "answer":        get(["answer", "label", "gt"], "").strip().lower(),
        "relation_type": get(["relation_type", "rel_type", "type"], "unknown"),
    }

rbench_norm = [normalize_entry(e) for e in rbench_data]
rbench_norm = [e for e in rbench_norm if e['question'] and e['answer'] in ('yes','no')]

# Show distribution
import collections
rel_counts = collections.Counter(e['relation_type'] for e in rbench_norm)
print(f"Total usable questions: {len(rbench_norm):,}")
print("By relation type:")
for rel, cnt in rel_counts.most_common():
    print(f"  {rel}: {cnt:,}")
print(f"\nSample: {rbench_norm[0]}")

In [ ]:
# ── Select 200 diverse images (balanced across relation types) ─────────────
import random
random.seed(42)
SUBSET_SIZE = 200

# Group questions by image_id
from collections import defaultdict
by_image = defaultdict(list)
for entry in rbench_norm:
    by_image[entry['image_id']].append(entry)

all_image_ids = list(by_image.keys())
print(f"Total unique images in R-Bench: {len(all_image_ids):,}")

# Try to pick images that have both spatial AND action/attribute questions
def has_variety(qlist):
    types = {q['relation_type'] for q in qlist}
    return len(types) >= 2

varied = [iid for iid in all_image_ids if has_variety(by_image[iid])]
fallback = [iid for iid in all_image_ids if not has_variety(by_image[iid])]

selected_ids = (random.sample(varied, min(SUBSET_SIZE, len(varied))) +
                random.sample(fallback, max(0, SUBSET_SIZE - len(varied))))[:SUBSET_SIZE]

print(f"Selected {len(selected_ids)} images ({len(varied)} with type variety)")

In [ ]:
# ── Download 200 nocaps images ─────────────────────────────────────────────
import requests
from tqdm.auto import tqdm
from pathlib import Path

EVAL_IMAGES_DIR = "/content/rbench_images"
os.makedirs(EVAL_IMAGES_DIR, exist_ok=True)

# Build a mapping: image_id → url (from R-Bench entries)
id_to_url = {}
for entry in rbench_norm:
    if entry['image_url']:
        id_to_url[entry['image_id']] = entry['image_url']

downloaded, skipped, failed = 0, 0, 0
image_id_to_path = {}  # populated below

for iid in tqdm(selected_ids, desc="Downloading images"):
    out_path = Path(f"{EVAL_IMAGES_DIR}/{iid}.jpg")
    if out_path.exists():
        skipped += 1
        image_id_to_path[iid] = str(out_path)
        continue

    url = id_to_url.get(iid, "")
    if not url:
        failed += 1
        continue

    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        out_path.write_bytes(resp.content)
        image_id_to_path[iid] = str(out_path)
        downloaded += 1
    except Exception as e:
        failed += 1

print(f"\n✅ Downloaded: {downloaded} | Already cached: {skipped} | Failed: {failed}")
print(f"Images available for eval: {len(image_id_to_path)}")

# ── Build eval dataset: list of (image_path, questions_for_image) ──────────
eval_dataset = [
    {
        "image_id":   iid,
        "image_path": image_id_to_path[iid],
        "questions":  by_image[iid],
    }
    for iid in selected_ids
    if iid in image_id_to_path
]
print(f"Eval dataset: {len(eval_dataset)} images with "
      f"{sum(len(d['questions']) for d in eval_dataset):,} questions")

# Save the subset info for reproducibility
subset_path = f"{REPO_DIR}/eval/rbench_subset.json"
with open(subset_path, 'w') as f:
    json.dump(eval_dataset, f, indent=2)
print(f"Subset saved → {subset_path}")

---
## Section 4: Baseline 1 — No Correction
Raw BLIP-2 answers to R-Bench Yes/No questions, no correction applied.
This is the lower bound for R-POPE.

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

print("Running Baseline 1: No Correction...")
baseline1_rows   = []  # per-question rows for CSV
baseline1_preds  = []  # for metric computation

for item in tqdm(eval_dataset, desc="Baseline 1"):
    image = Image.open(item['image_path']).convert("RGB")

    for q in item['questions']:
        pred = vqa_probe(image, q['question'])
        gt   = q['answer']
        baseline1_preds.append({'pred': pred, 'gt': gt})
        baseline1_rows.append({
            'image_id':      item['image_id'],
            'question':      q['question'],
            'gt':            gt,
            'pred':          pred,
            'correct':       (pred == gt),
            'relation_type': q['relation_type'],
        })

df_b1 = pd.DataFrame(baseline1_rows)
B1_CSV = f"{REPO_DIR}/eval/baseline_no_correction.csv"
df_b1.to_csv(B1_CSV, index=False)

b1_metrics = compute_rpope_metrics(baseline1_preds)
print(f"\n{'='*50}")
print("BASELINE 1 (No Correction) — R-POPE Metrics")
print(f"{'='*50}")
for k, v in b1_metrics.items():
    if isinstance(v, float): print(f"  {k:12s}: {v:.4f}")
    else:                    print(f"  {k:12s}: {v}")
print(f"Results saved → {B1_CSV}")

---
## Section 5: Baseline 2 — Self-Refinement
Ask BLIP-2 to re-examine its own description, then re-run VQA on the refined caption.

In [ ]:
from triple_extractor import TripleExtractor
extractor = TripleExtractor()

print("Running Baseline 2: Self-Refinement...")
baseline2_rows  = []
baseline2_preds = []

def self_refine_caption(image: Image.Image) -> str:
    """Ask BLIP-2 to generate a caption, then ask it to self-correct."""
    # Step 1: initial caption
    caption = generate_caption(image)
    # Step 2: ask BLIP-2 to refine
    refine_prompt = (
        f"Initial description: {caption}\n"
        "Look at the image carefully. Is the above description accurate? "
        "If anything is wrong about the relationships between objects, correct only those parts."
    )
    inputs = blip2_processor(images=image, text=refine_prompt, return_tensors="pt").to(
        blip2_model.device, torch.float16)
    with torch.no_grad():
        ids = blip2_model.generate(**inputs, max_new_tokens=100)
    refined = blip2_processor.decode(ids[0], skip_special_tokens=True).strip()
    # If the model just echoes the prompt or says "yes", fall back to original
    if len(refined) < 20 or refined.lower().startswith("yes"):
        return caption
    return refined

for item in tqdm(eval_dataset, desc="Baseline 2"):
    image = Image.open(item['image_path']).convert("RGB")
    refined_caption = self_refine_caption(image)

    for q in item['questions']:
        pred = vqa_probe(image, q['question'])
        gt   = q['answer']
        baseline2_preds.append({'pred': pred, 'gt': gt})
        baseline2_rows.append({
            'image_id':       item['image_id'],
            'question':       q['question'],
            'refined_caption': refined_caption,
            'gt':             gt,
            'pred':           pred,
            'correct':        (pred == gt),
            'relation_type':  q['relation_type'],
        })

df_b2 = pd.DataFrame(baseline2_rows)
B2_CSV = f"{REPO_DIR}/eval/baseline_self_refine.csv"
df_b2.to_csv(B2_CSV, index=False)

b2_metrics = compute_rpope_metrics(baseline2_preds)
print(f"\n{'='*50}")
print("BASELINE 2 (Self-Refinement) — R-POPE Metrics")
print(f"{'='*50}")
for k, v in b2_metrics.items():
    if isinstance(v, float): print(f"  {k:12s}: {v:.4f}")
    else:                    print(f"  {k:12s}: {v}")
print(f"Results saved → {B2_CSV}")

---
## Section 6: Full RelCheck
Runs the complete 3-stage pipeline on all 200 images.
For each R-Bench question, uses verified triples where available, falls back to VQA otherwise.

In [ ]:
import Levenshtein
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def answer_question_with_relcheck(result, image, question: str) -> str:
    """
    Given a RelCheckResult and a R-Bench question, return 'yes' or 'no'.
    Tries to match the question against verified triples first.
    Falls back to VQA if no match.
    """
    q_lower = question.lower()
    for triple in result.triples:
        if triple.hallucinated is None:
            continue
        s, r, o = triple.subject.lower(), triple.relation.lower(), triple.obj.lower()
        # Match if at least 2 of 3 triple components appear in the question
        matches = sum([
            any(w in q_lower for w in s.split()),
            r in q_lower,
            any(w in q_lower for w in o.split()),
        ])
        if matches >= 2:
            return "no" if triple.hallucinated else "yes"
    # No matched triple → fall back to VQA on corrected caption context
    return vqa_probe(image, question)


print("Running Full RelCheck on 200 images...")
relcheck_rows  = []
relcheck_preds = []
pipeline_results_cache = {}

smoother = SmoothingFunction().method1

for item in tqdm(eval_dataset, desc="RelCheck"):
    image = Image.open(item['image_path']).convert("RGB")
    try:
        result = pipeline.run(
            image_path      = item['image_path'],
            blip2_processor = blip2_processor,
            blip2_model     = blip2_model,
        )
        pipeline_results_cache[item['image_id']] = result
    except Exception as e:
        print(f"Pipeline error on {item['image_id']}: {e}")
        continue

    # BLEU-4: compare corrected vs original
    ref_tokens  = result.original_caption.split()
    hyp_tokens  = result.corrected_caption.split()
    bleu4 = sentence_bleu([ref_tokens], hyp_tokens,
                           weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother)

    for q in item['questions']:
        pred = answer_question_with_relcheck(result, image, q['question'])
        gt   = q['answer']
        relcheck_preds.append({'pred': pred, 'gt': gt})
        relcheck_rows.append({
            'image_id':          item['image_id'],
            'question':          q['question'],
            'gt':                gt,
            'pred':              pred,
            'correct':           (pred == gt),
            'relation_type':     q['relation_type'],
            'original_caption':  result.original_caption,
            'corrected_caption': result.corrected_caption,
            'n_triples':         result.n_triples,
            'n_hallucinated':    result.n_hallucinated,
            'edit_rate':         result.edit_rate,
            'bleu4':             bleu4,
        })

df_rc = pd.DataFrame(relcheck_rows)
RC_CSV = f"{REPO_DIR}/eval/relcheck_results.csv"
df_rc.to_csv(RC_CSV, index=False)

rc_metrics = compute_rpope_metrics(relcheck_preds)
print(f"\n{'='*50}")
print("FULL RELCHECK — R-POPE Metrics")
print(f"{'='*50}")
for k, v in rc_metrics.items():
    if isinstance(v, float): print(f"  {k:12s}: {v:.4f}")
    else:                    print(f"  {k:12s}: {v}")
print(f"\nAvg edit rate: {df_rc.groupby('image_id')['edit_rate'].first().mean():.4f}")
print(f"Avg BLEU-4:    {df_rc.groupby('image_id')['bleu4'].first().mean():.4f}")
print(f"Results saved → {RC_CSV}")

---
## Section 7: Ablation Study
Three ablated variants to show each module contributes independently:
1. **No Spatial Verifier** — OWL-ViT disabled, VQA handles all relations
2. **No VQA Verifier** — BLIP-2 VQA disabled, spatial-only
3. **Detect-Only** — Runs stages 1+2 but no correction (stage 3 skipped)

In [ ]:
ablation_configs = [
    dict(name="no_spatial",   skip_spatial=True,  skip_vqa=False, detection_only=False),
    dict(name="no_vqa",       skip_spatial=False, skip_vqa=True,  detection_only=False),
    dict(name="detect_only",  skip_spatial=False, skip_vqa=False, detection_only=True),
]

ablation_summary = []

for config in ablation_configs:
    name = config.pop("name")
    print(f"\n{'='*50}")
    print(f"Running ablation: {name}")
    print(f"{'='*50}")

    abl_pipeline = RelCheckPipeline(
        together_api_key = TOGETHER_API_KEY,
        **config,
    )

    abl_rows  = []
    abl_preds = []

    for item in tqdm(eval_dataset, desc=name):
        image = Image.open(item['image_path']).convert("RGB")
        try:
            result = abl_pipeline.run(
                image_path      = item['image_path'],
                blip2_processor = blip2_processor,
                blip2_model     = blip2_model,
            )
        except Exception as e:
            print(f"  Error on {item['image_id']}: {e}")
            continue

        for q in item['questions']:
            pred = answer_question_with_relcheck(result, image, q['question'])
            gt   = q['answer']
            abl_preds.append({'pred': pred, 'gt': gt})
            abl_rows.append({
                'ablation':       name,
                'image_id':       item['image_id'],
                'question':       q['question'],
                'gt':             gt,
                'pred':           pred,
                'correct':        (pred == gt),
                'relation_type':  q['relation_type'],
                'n_hallucinated': result.n_hallucinated,
                'edit_rate':      result.edit_rate,
            })

    metrics = compute_rpope_metrics(abl_preds)
    print(f"  Accuracy={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}")
    ablation_summary.append({'variant': name, **metrics})

    df_abl = pd.DataFrame(abl_rows)
    csv_path = f"{REPO_DIR}/eval/ablation_{name}.csv"
    df_abl.to_csv(csv_path, index=False)
    print(f"  Saved → {csv_path}")

# Add full RelCheck as the last row
ablation_summary.append({'variant': 'RelCheck (full)', **rc_metrics})

df_ablation = pd.DataFrame(ablation_summary)
ABL_CSV = f"{REPO_DIR}/eval/ablation_results.csv"
df_ablation.to_csv(ABL_CSV, index=False)
print(f"\n✅ Ablation summary saved → {ABL_CSV}")
display(df_ablation[['variant','accuracy','precision','recall','f1']].round(4))

---
## Section 8: R-CHAIR Annotation (50 images)
Manually annotate whether each extracted triple is a hallucination.
Computes R-CHAIR_s (caption-level) and R-CHAIR_i (triple-level) metrics.

**This section requires 30–60 minutes of manual annotation.**
Each cell presents one image + its extracted triples; you enter 0 (correct) or 1 (hallucinated) for each.

In [ ]:
from IPython.display import display as ipy_display, Image as IPyImage
import ipywidgets as widgets

# Use first 50 images from eval set
RCHAIR_SUBSET = eval_dataset[:50]
rchair_labels = {}  # image_id → list of {triple_str, hallucinated}

# Pre-populate using pipeline results already computed
for item in RCHAIR_SUBSET:
    iid = item['image_id']
    if iid in pipeline_results_cache:
        result = pipeline_results_cache[iid]
        rchair_labels[iid] = [
            {
                'triple': str(t),
                'relcheck_judgment': t.hallucinated,
                'human_judgment': None,  # to be filled below
            }
            for t in result.triples
        ]

print(f"R-CHAIR annotation ready for {len(RCHAIR_SUBSET)} images.")
print(f"{sum(len(v) for v in rchair_labels.values())} triples to annotate.")
print("\nRun the next cell to start the annotation interface.")

In [ ]:
# ── Interactive annotation loop ────────────────────────────────────────────
# For each image, shows the image + triples and asks you to confirm/override
# RelCheck's judgment.
#
# Press Enter to accept RelCheck's judgment, or type 'y'/'n' to override.

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

annotated_count = 0

for item in RCHAIR_SUBSET:
    iid = item['image_id']
    if iid not in rchair_labels:
        continue

    triples_info = rchair_labels[iid]
    if not triples_info:
        continue

    # Show image
    fig, ax = plt.subplots(figsize=(5,5))
    ax.imshow(mpimg.imread(item['image_path']))
    ax.axis('off')
    ax.set_title(f"Image: {iid}", fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"\nCaption: {pipeline_results_cache[iid].original_caption}")
    print("Triples (RelCheck judgment shown):")

    for i, t_info in enumerate(triples_info):
        rc_j = t_info['relcheck_judgment']
        rc_label = "HALLUCINATED" if rc_j else ("verified" if rc_j is False else "uncertain")
        resp = input(f"  [{i+1}] {t_info['triple']} — RelCheck says: {rc_label}. "
                     f"Hallucinated? (y/n/Enter=agree): ").strip().lower()
        if resp == 'y':
            t_info['human_judgment'] = True
        elif resp == 'n':
            t_info['human_judgment'] = False
        else:
            t_info['human_judgment'] = rc_j  # agree with RelCheck

    annotated_count += 1
    if annotated_count % 10 == 0:
        print(f"\n  [Progress: {annotated_count}/50 images annotated]\n")

print(f"\n✅ Annotation complete for {annotated_count} images.")

In [ ]:
# ── Compute R-CHAIR metrics ────────────────────────────────────────────────
rchair_rows = []
captions_with_hallu  = 0
total_captions       = 0
hallucinated_triples = 0
total_triples        = 0

for item in RCHAIR_SUBSET:
    iid = item['image_id']
    if iid not in rchair_labels:
        continue

    triples_info = rchair_labels[iid]
    judged = [t for t in triples_info if t['human_judgment'] is not None]
    if not judged:
        continue

    total_captions  += 1
    total_triples   += len(judged)
    n_hallu = sum(1 for t in judged if t['human_judgment'] is True)
    hallucinated_triples += n_hallu
    if n_hallu > 0:
        captions_with_hallu += 1

    for t_info in judged:
        rchair_rows.append({
            'image_id':        iid,
            'triple':          t_info['triple'],
            'relcheck_judgment': t_info['relcheck_judgment'],
            'human_judgment':  t_info['human_judgment'],
        })

rchair_s = captions_with_hallu / total_captions if total_captions else 0
rchair_i = hallucinated_triples / total_triples if total_triples else 0

df_rchair = pd.DataFrame(rchair_rows)
RCHAIR_CSV = f"{REPO_DIR}/eval/rchair_labels.csv"
df_rchair.to_csv(RCHAIR_CSV, index=False)

print(f"\n{'='*50}")
print("R-CHAIR METRICS")
print(f"{'='*50}")
print(f"  R-CHAIR_s (caption-level): {rchair_s:.4f}  "
      f"({captions_with_hallu}/{total_captions} captions have ≥1 hallucination)")
print(f"  R-CHAIR_i (triple-level):  {rchair_i:.4f}  "
      f"({hallucinated_triples}/{total_triples} triples hallucinated)")
print(f"\nLabels saved → {RCHAIR_CSV}")

---
## Section 9: Results Tables & Figures
Generates publication-ready figures saved to `figures/`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Figure 1: Main results bar chart (R-POPE accuracy + F1) ───────────────
conditions = ["No Correction", "Self-Refinement", "RelCheck (ours)"]
accuracy   = [b1_metrics['accuracy'], b2_metrics['accuracy'], rc_metrics['accuracy']]
f1         = [b1_metrics['f1'],       b2_metrics['f1'],       rc_metrics['f1']]
precision  = [b1_metrics['precision'],b2_metrics['precision'],rc_metrics['precision']]
recall     = [b1_metrics['recall'],   b2_metrics['recall'],   rc_metrics['recall']]

x = np.arange(len(conditions))
width = 0.2
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(9, 5))
bars_acc  = ax.bar(x - 1.5*width, accuracy,  width, label='Accuracy',  color=colors[0])
bars_prec = ax.bar(x - 0.5*width, precision, width, label='Precision', color=colors[1])
bars_rec  = ax.bar(x + 0.5*width, recall,    width, label='Recall',    color=colors[2])
bars_f1   = ax.bar(x + 1.5*width, f1,        width, label='F1',        color=colors[3])

ax.set_xlabel('Method', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('R-POPE: Relational Hallucination Detection\n(200-image R-Bench subset)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(conditions, fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars_acc, bars_prec, bars_rec, bars_f1]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=7)

plt.tight_layout()
fig.savefig(f"{REPO_DIR}/figures/rpope_results.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved → figures/rpope_results.png")

In [ ]:
# ── Figure 2: Ablation Study ───────────────────────────────────────────────
df_abl_plot = pd.read_csv(ABL_CSV)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

variants = df_abl_plot['variant'].tolist()
colors_abl = ['#8172B3', '#64B5CD', '#CCB974', '#4C72B0']

# Accuracy
axes[0].bar(variants, df_abl_plot['accuracy'], color=colors_abl)
axes[0].set_title('Ablation: R-POPE Accuracy', fontsize=12)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1.0)
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_abl_plot['accuracy']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

# F1
axes[1].bar(variants, df_abl_plot['f1'], color=colors_abl)
axes[1].set_title('Ablation: R-POPE F1', fontsize=12)
axes[1].set_ylabel('F1 Score')
axes[1].set_ylim(0, 1.0)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_abl_plot['f1']):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Ablation Study — RelCheck Module Contributions', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(f"{REPO_DIR}/figures/ablation_results.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved → figures/ablation_results.png")

In [ ]:
# ── Figure 3: R-POPE Accuracy broken down by relation type ────────────────
df_rc_rt = pd.read_csv(RC_CSV)
df_b1_rt = pd.read_csv(B1_CSV)

rel_types = df_rc_rt['relation_type'].unique()
rc_by_rel = df_rc_rt.groupby('relation_type')['correct'].mean()
b1_by_rel = df_b1_rt.groupby('relation_type')['correct'].mean()

common_rels = sorted(set(rc_by_rel.index) & set(b1_by_rel.index))
x = np.arange(len(common_rels))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, [b1_by_rel[r] for r in common_rels], w,
       label='No Correction', color='#4C72B0')
ax.bar(x + w/2, [rc_by_rel[r] for r in common_rels], w,
       label='RelCheck (ours)', color='#DD8452')

ax.set_xticks(x)
ax.set_xticklabels([r.capitalize() for r in common_rels], fontsize=11)
ax.set_ylabel('R-POPE Accuracy')
ax.set_title('R-POPE Accuracy by Relation Type', fontsize=13)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(f"{REPO_DIR}/figures/accuracy_by_relation.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved → figures/accuracy_by_relation.png")

In [ ]:
# ── Figure 4: Qualitative before/after examples ───────────────────────────
# Pick 5 images where RelCheck actually made a correction

corrected_examples = [
    (iid, res)
    for iid, res in pipeline_results_cache.items()
    if res.n_corrected > 0 and res.original_caption != res.corrected_caption
][:5]

if len(corrected_examples) < 5:
    print(f"Only {len(corrected_examples)} corrected examples found.")

n = len(corrected_examples)
if n == 0:
    print("No corrections found to display. Try running on more images.")
else:
    fig, axes = plt.subplots(n, 1, figsize=(12, 4 * n))
    if n == 1:
        axes = [axes]

    for ax, (iid, res) in zip(axes, corrected_examples):
        img_path = next(
            (d['image_path'] for d in eval_dataset if d['image_id'] == iid),
            None
        )
        if img_path and os.path.exists(img_path):
            ax.imshow(mpimg.imread(img_path))
        ax.axis('off')
        title = (
            f"ORIGINAL:  {res.original_caption}\n"
            f"CORRECTED: {res.corrected_caption}"
        )
        ax.set_title(title, fontsize=9, loc='left', pad=8, wrap=True)

    plt.suptitle('RelCheck: Qualitative Before/After Examples', fontsize=13, y=1.01)
    plt.tight_layout()
    fig.savefig(f"{REPO_DIR}/figures/qualitative_examples.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved → figures/qualitative_examples.png")

In [ ]:
# ── Final Summary Table ────────────────────────────────────────────────────
print("\n" + "="*70)
print("RELCHECK — FINAL RESULTS SUMMARY")
print("="*70)

summary = pd.DataFrame([
    {"Method": "No Correction",    **{k: round(v,4) for k,v in b1_metrics.items() if isinstance(v,float)}},
    {"Method": "Self-Refinement",  **{k: round(v,4) for k,v in b2_metrics.items() if isinstance(v,float)}},
    {"Method": "RelCheck (ours)",  **{k: round(v,4) for k,v in rc_metrics.items() if isinstance(v,float)}},
])[['Method', 'accuracy', 'precision', 'recall', 'f1', 'yes_ratio']]

display(summary)

print(f"\nAdditional RelCheck metrics:")
df_rc_img = df_rc.groupby('image_id').first().reset_index()
print(f"  Avg Edit Rate:   {df_rc_img['edit_rate'].mean():.4f}")
print(f"  Avg BLEU-4:      {df_rc_img['bleu4'].mean():.4f}")
if total_captions > 0:
    print(f"  R-CHAIR_s:       {rchair_s:.4f}")
    print(f"  R-CHAIR_i:       {rchair_i:.4f}")

print(f"\nFigures saved to: {REPO_DIR}/figures/")
print(f"CSVs saved to:    {REPO_DIR}/eval/")
print("\n✅ All done! Push results to GitHub:")
print(f"  cd {REPO_DIR} && git add eval/ figures/ && "
      f"git commit -m 'Add evaluation results' && git push")